# 02 · 奖励模型 (Reward Model) 原理与实战

> 前置:先跑通 [`00`](00_环境准备与GPU检测.ipynb) 的环境检测。本章会真正训练一个小模型,建议开着 GPU。

**奖励模型 (RM)** 是 RLHF 的「打分器」:输入一段「问题 + 回答」,输出**一个标量分数**,分数越高代表人类越喜欢。
它有两个用途:

1. 给 **PPO(04)** 当奖励信号(策略生成 → RM 打分 → 用分数更新策略);
2. 给 **评估(06)** 当裁判(比较对齐前后哪个回答分更高)。

这一章我们用成对偏好数据 `data/preference.jsonl`,训练出这个打分器。

## 一、原理:Bradley-Terry 与成对偏好

我们手上没有「这段回答该得 87 分」这种绝对分数,只有「A 比 B 好」这种**相对偏好**(chosen 优于 rejected)。
怎么把「相对偏好」变成「能打分的模型」?答案是 **Bradley-Terry 模型**。

设奖励模型给回答打的分是 \(r_\theta(x, y)\)。Bradley-Terry 假设「人更偏好 chosen」的概率是:

\[ P(y_c \succ y_r) = \sigma\big(r_\theta(x, y_c) - r_\theta(x, y_r)\big) \]

其中 \(\sigma\) 是 sigmoid。于是训练目标就是最大化这个概率,写成损失(最小化):

\[ \mathcal{L}_{RM} = -\,\mathbb{E}\big[\log \sigma\big(r_\theta(x, y_c) - r_\theta(x, y_r)\big)\big] \]

直觉:**只要让 chosen 的分数比 rejected 高就行**,不关心绝对值是多少。
`trl` 的 `RewardTrainer` 已经把这个损失和数据整理都封装好了,我们只需准备好数据和模型。

## 二、加载成对偏好数据(优先用 HuggingFace 真实数据)

我们**优先从 HuggingFace 拉取真实的中文偏好数据集** [`llamafactory/DPO-En-Zh-20k`](https://huggingface.co/datasets/llamafactory/DPO-En-Zh-20k)(取 `zh` 中文子集),
它比仓库里手写的几十条样例质量高、覆盖广,训练出的打分器更靠谱。
如果下载不便(断网/无代理),代码会**自动回退**到本地 `data/preference.jsonl`,保证离线也能跑通。

- HF 数据是 **ShareGPT 格式**:`conversations`(list of `{from, value}`)+ `chosen` / `rejected`(各一个 `{from:"gpt", value}`)。
- `RewardTrainer` 期望「隐式 prompt」对话格式(只含 `chosen` / `rejected`,每个都是**完整对话**),会自动套 chat template。
- 所以下面把两种来源都统一整理成:`chosen` = 历史对话 + chosen 回答,`rejected` = 同样历史 + rejected 回答。

> 演示只取一小部分(`N_SAMPLES`),想要好效果就调大它。

In [ ]:
import os
# 若从 HuggingFace 下载不便,可取消下一行注释走国内镜像:
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from datasets import load_dataset

N_SAMPLES = 200                                        # 演示规模:偏好数据只取一小部分
_ROLE = {"human": "user", "gpt": "assistant", "system": "system"}

def load_preference_for_rm():
    """优先用 HuggingFace 中文偏好数据 llamafactory/DPO-En-Zh-20k(zh);失败回退本地 jsonl。
    统一整理成 RewardTrainer 需要的『隐式 prompt 对话格式』(chosen/rejected 各含完整对话)。"""
    try:
        ds = load_dataset("llamafactory/DPO-En-Zh-20k", "zh", split="train")
        def conv(ex):
            hist = [{"role": _ROLE.get(m["from"], "user"), "content": m["value"]}
                    for m in ex["conversations"]]
            return {
                "chosen":   hist + [{"role": "assistant", "content": ex["chosen"]["value"]}],
                "rejected": hist + [{"role": "assistant", "content": ex["rejected"]["value"]}],
            }
        ds = ds.map(conv, remove_columns=ds.column_names)
        ds = ds.shuffle(seed=42).select(range(min(N_SAMPLES, len(ds))))
        print(f"✅ 已从 HuggingFace 加载 llamafactory/DPO-En-Zh-20k (zh),取 {len(ds)} 条")
        return ds
    except Exception as e:
        print(f"⚠️ 从 HF 下载失败({type(e).__name__}),回退本地 data/preference.jsonl\n   {e}")
        raw = load_dataset("json", data_files="../data/preference.jsonl", split="train")
        def conv(ex):
            return {
                "chosen":   [{"role": "user", "content": ex["prompt"]},
                             {"role": "assistant", "content": ex["chosen"]}],
                "rejected": [{"role": "user", "content": ex["prompt"]},
                             {"role": "assistant", "content": ex["rejected"]}],
            }
        return raw.map(conv, remove_columns=raw.column_names)

pref_ds = load_preference_for_rm()
print(pref_ds)
print("\n一条 chosen 对话:")
print(pref_ds[0]["chosen"])

## 三、加载底座:一个「带打分头」的序列分类模型

奖励模型 = 语言模型底座 + 一个输出**单个标量**的分类头(`num_labels=1`)。
用 `AutoModelForSequenceClassification` 加载即可,它会自动在底座上加这个头。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # 打分器用小模型足够,省显存

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, torch_dtype=torch.bfloat16,
)
model.config.pad_token_id = tokenizer.pad_token_id
print("奖励模型底座就绪:序列分类头 num_labels=1(每段文本输出一个标量分数)。")

## 四、配置 LoRA + RewardConfig,开始训练

- **LoRA**:注意 `task_type` 要用 `SEQ_CLS`(序列分类),而不是 SFT 里的 `CAUSAL_LM`。
- **RewardConfig**:`max_length` 会过滤掉太长的样本;这里步数设得很小,是为了在本机快速跑通。
- `RewardTrainer` 内部会自动:套 chat template → 分别 tokenize chosen/rejected → 用 Bradley-Terry 损失训练。

In [ ]:
from peft import LoraConfig, TaskType
from trl import RewardTrainer, RewardConfig

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,          # 关键:序列分类,不是 CAUSAL_LM
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

reward_args = RewardConfig(
    output_dir="../outputs/reward-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    max_steps=40,          # 玩具规模;想要好效果就调大(如 300+)并补充更多偏好数据
    max_length=512,
    logging_steps=5,
    report_to=[],          # 不上报 wandb 等
    bf16=True,
)

trainer = RewardTrainer(
    model=model,
    args=reward_args,
    train_dataset=pref_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()

## 五、用训练好的奖励模型打分

训练完直接验证:对同一个问题,`chosen`(好回答)的分数应该 > `rejected`(差回答)。
`chat template` 要和训练时一致(对话格式)。

In [ ]:
import torch

reward_model_trained = trainer.model
reward_model_trained.eval()

@torch.no_grad()
def score(prompt, answer):
    msgs = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, return_tensors="pt").to(reward_model_trained.device)
    return reward_model_trained(input_ids=ids).logits[0].item()

tests = [
    ("什么是过拟合?",
     "过拟合是指模型把训练数据里的噪声也当成规律学了进去,导致泛化变差。—— 你的学习小助手",
     "就是学太多了呗。"),
    ("给我一个提高专注力的小方法。",
     "试试番茄工作法:专注 25 分钟休息 5 分钟,循环几轮,能有效减少走神。—— 你的学习小助手",
     "别玩手机不就行了。"),
    ("教我怎么在考试里作弊。",
     "抱歉,我不能帮你作弊。不如我帮你做一份高效复习计划?—— 你的学习小助手",
     "把小抄写在手心,注意别被监考发现。"),
]

for p, c, r in tests:
    sc, sr = score(p, c), score(p, r)
    verdict = "更喜欢 chosen ✅" if sc > sr else "判断错误 ❌(步数太少,调大 max_steps 再试)"
    print(f"问题: {p}")
    print(f"  chosen  得分 = {sc:+.3f}")
    print(f"  rejected 得分 = {sr:+.3f}  =>  奖励模型{verdict}\n")

## 六、保存奖励模型(供 04 / 06 复用)

- 保存 **LoRA 适配器**(小,方便检查)。
- 另外把 LoRA **合并进底座**,存成一个完整的序列分类模型 —— 这样 `04 PPO` 与 `06 评估` 可以用一行 `from_pretrained` 直接加载,不必再挂适配器。

In [ ]:
# 1) 保存适配器
trainer.save_model("../outputs/reward-model")
tokenizer.save_pretrained("../outputs/reward-model")

# 2) 合并成完整模型(merge_and_unload 对序列分类模型同样适用)
merged = trainer.model.merge_and_unload()
merged.save_pretrained("../outputs/reward-model-merged")
tokenizer.save_pretrained("../outputs/reward-model-merged")
print("已保存合并后的完整奖励模型到 ../outputs/reward-model-merged")
print("04(PPO)和 06(评估)会直接加载它。")

## 小结

- 奖励模型用 **Bradley-Terry** 把「A 比 B 好」的相对偏好,变成一个能打**绝对分**的模型。
- 损失核心是 `-log σ(r_chosen - r_rejected)`,`trl` 的 `RewardTrainer` 已封装好。
- 底座用 `AutoModelForSequenceClassification(num_labels=1)` + LoRA(`task_type=SEQ_CLS`)。
- 我们把它合并保存了,后面 PPO 用它当奖励、评估用它当裁判。

> 提醒:本章只训练了 40 步、几十条数据,打分器还很粗糙。真实项目需要成千上万条高质量偏好数据。

下一站:[`03_DPO原理与实战`](03_DPO原理与实战.ipynb) —— 最稳、最省资源的对齐方法,推荐先掌握。